# 01_dataprep
**CSE570 Skill Based Assignment.**


## 2. Data Cleaning


### 2.1 Missing-value summary & treatment
Only `ca` and `thal` contain missing values. Since only ~2% of rows are affected, we drop them — simple and safe for a 303-row dataset.

In [ ]:
print(df.isna().sum())
print(f'\nRows with any missing value: {df.isna().any(axis=1).sum()} ({df.isna().any(axis=1).mean()*100:.1f}%)')

df = df.dropna().reset_index(drop=True)
print(f'\nShape after dropping missing rows: {df.shape}')

### 2.2 Duplicate handling


In [ ]:
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups:
    df = df.drop_duplicates().reset_index(drop=True)

### 2.3 Datatype correction
`ca` and `thal` arrive as floats only because of NaN; they are categorical codes and are cast to the correct dtype.

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'num']
df[cat_cols] = df[cat_cols].astype(int)

for c in ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']:
    df[c] = df[c].astype('category')

df.info()

### 2.4 Target binarization
`num` = 0 means no disease; 1-4 are increasing severity levels. Following standard practice we binarize: 0 = absence, 1 = presence.

In [ ]:
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns='num')

num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True).round(3))

### 2.5 Outlier inspection
Numeric features: age, trestbps, chol, thalach, oldpeak. We inspect with boxplots and the IQR rule. Extreme trestbps/chol values are physiologically possible, so we keep them but report them.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, c in zip(axes, num_cols):
    df[c].plot.box(ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.savefig('outliers_boxplot.png', dpi=150)
plt.show()

for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    print(f'{c}: {n_out} outliers ({n_out/len(df)*100:.1f}%)')